# Smoke test: does the code hold water?

Validates the fixes applied on 2026-06-12 and the core numerical claims of Paper 1:

1. `import src` works (new `src/of.py`, `src/make_weights.py`)
2. `gls_amplitude` reproduces `OptimumFilter.fit` exactly (weight-convention match)
3. Rank-1 EMPCA ≡ OF (Experiment A core claim)
4. `solve_eigvec_full` now accepts 1D weights (regression) and is ≥ fast mode in χ²
5. `w_orthonormalize` enforces the Bridge-Theorem gauge P†Σ⁻¹P = I
6. Noise-generation ↔ `PSDCalculator` normalization round-trip (Fig 7 Brownian 184× bug check)

Run top-to-bottom; the last cell prints a PASS/FAIL summary. Equivalent pytest suite: `tests/` (run `PYTHONPATH=. pytest tests -q`).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

results = {}  # check name -> (passed: bool, detail: str)

## Check 1 — package imports

In [ ]:
try:
    import src
    from src.OptimumFilter import OptimumFilter
    from src.PSDCalculator import calculate_psd
    from src.of import gls_amplitude, project_rank1
    from src.make_weights import build_of_one_sided_weights, make_inverse_psd_weights
    from src.EMPCA.empca_TCY_optimized import empca_solver, w_orthonormalize
    from src.EMPCA.empca_equivalence_utils import (
        fit_empca_no_smoothing, phase_align_basis, weighted_cosine)
    results['1 import src'] = (True, f"exports: {src.__all__}")
except Exception as e:
    results['1 import src'] = (False, repr(e))
    raise
print(results['1 import src'])

## Synthetic data

Same construction as `tests/conftest.py`: exponential-pulse template, pink (1/f) noise
with a known one-sided PSD, amplitudes drawn uniformly in [0.5, 1.5].

In [ ]:
N, FS, NTR = 1024, 1.0e5, 256
rng = np.random.default_rng(7)

t = np.arange(N) / FS
s = np.exp(-t / 8e-3) - np.exp(-t / 8e-4)
s /= s.max()

f = np.fft.rfftfreq(N, d=1.0 / FS)
J = np.empty_like(f)
J[1:] = 1e-4 * (f[1] / f[1:])
J[0] = J[1]

def colored_noise(rng, J, n, fs, n_traces):
    n_rfft = n // 2 + 1
    scale = np.sqrt(J * fs * n / 2.0)
    re = rng.standard_normal((n_traces, n_rfft))
    im = rng.standard_normal((n_traces, n_rfft))
    X = (re + 1j * im) / np.sqrt(2.0) * scale[None, :]
    X[:, 0] = re[:, 0] * scale[0]
    if n % 2 == 0:
        X[:, -1] = re[:, -1] * scale[-1]
    return np.fft.irfft(X, n=n, axis=1)

amps = rng.uniform(0.5, 1.5, size=NTR)
noise = colored_noise(rng, J, N, FS, NTR)
traces = amps[:, None] * s[None, :] + noise

w = build_of_one_sided_weights(J, N)
s_f = np.fft.rfft(s)
X_f = np.fft.rfft(traces, axis=1)

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(t * 1e3, traces[0], lw=0.5, label='trace')
ax[0].plot(t * 1e3, amps[0] * s, 'r', label='clean')
ax[0].set_xlabel('time (ms)'); ax[0].legend()
ax[1].loglog(f[1:], J[1:]); ax[1].set_xlabel('f (Hz)'); ax[1].set_title('injected PSD')
plt.tight_layout(); plt.show()

## Check 2 — `gls_amplitude` ≡ `OptimumFilter.fit`

In [ ]:
of = OptimumFilter(s, J, FS)
amp_of = np.array([of.fit(tr)[0] for tr in traces])
amp_gls = gls_amplitude(X_f, s_f, w)
max_rel = np.max(np.abs(amp_gls - amp_of) / np.abs(amp_of))
ok = max_rel < 1e-8
results['2 gls_amplitude == OF.fit'] = (ok, f'max rel diff = {max_rel:.3e}')
print(results['2 gls_amplitude == OF.fit'])

## Check 3 — rank-1 EMPCA ≡ OF (Experiment A)

In [ ]:
eigvec1, coeff1, chi2s1 = fit_empca_no_smoothing(X_f, w, n_comp=1, n_iter=100)
u = phase_align_basis(eigvec1[0], s_f, w)
cos = weighted_cosine(u, s_f, w)
r = abs(np.corrcoef(np.abs(coeff1[:, 0]), amp_of)[0, 1])
ok = (cos > 0.99) and (r > 0.999)
results['3 rank-1 EMPCA == OF'] = (ok, f'weighted cosine = {cos:.6f}, corr(coeff, OF amp) = {r:.6f}')
print(results['3 rank-1 EMPCA == OF'])

plt.figure(figsize=(4, 4))
plt.scatter(amp_of, np.abs(coeff1[:, 0]), s=8)
plt.xlabel('OF amplitude'); plt.ylabel('|EMPCA rank-1 coeff|')
plt.title(f'r = {r:.6f}'); plt.tight_layout(); plt.show()

## Check 4 — rank-2: fixed `solve_eigvec_full` with 1D weights, fast vs full

Data now includes pulse-shape variation (timing-derivative admixture), the regime of Experiment E.

In [ ]:
ds = np.gradient(s); ds = ds / np.linalg.norm(ds) * np.linalg.norm(s)
b = rng.normal(0.0, 0.05, size=NTR)
traces2 = amps[:, None] * s[None, :] + (amps * b)[:, None] * ds[None, :] + noise
X2_f = np.fft.rfft(traces2, axis=1)

# regression: 1D weights must not crash the full solver
try:
    solver = empca_solver(2, X2_f, w)
    eig = solver.solve_eigvec_full()
    crash_ok = np.all(np.isfinite(eig.real)) and np.allclose(eig[:, w == 0], 0.0)
except Exception as e:
    crash_ok, eig = False, repr(e)
results['4a full solver accepts 1D weights'] = (bool(crash_ok), str(type(eig)))
print(results['4a full solver accepts 1D weights'])

eig_fast, _, chi2_fast = fit_empca_no_smoothing(X2_f, w, n_comp=2, n_iter=150, patience=20, mode='fast')
eig_full, _, chi2_full = fit_empca_no_smoothing(X2_f, w, n_comp=2, n_iter=150, patience=20, mode='full')
ok = min(chi2_full) <= min(chi2_fast) * (1 + 1e-6)
results['4b full chi2 <= fast chi2'] = (ok, f'fast = {min(chi2_fast):.6g}, full = {min(chi2_full):.6g}')
print(results['4b full chi2 <= fast chi2'])

# weighted principal angles between fast and full rank-2 subspaces
Qa = w_orthonormalize(eig_fast, w)
Qb = w_orthonormalize(eig_full, w)
sv = np.clip(np.linalg.svd((Qa * w[None, :]) @ Qb.conj().T, compute_uv=False), 0, 1)
angles = np.degrees(np.arccos(sv))
ok = angles.max() < 5.0
results['4c fast/full subspace agreement'] = (ok, f'principal angles (deg) = {np.round(angles, 3)}')
print(results['4c fast/full subspace agreement'])

plt.figure(figsize=(5, 3))
plt.plot(chi2_fast, label='fast (decoupled M-step)')
plt.plot(chi2_full, label='full (exact M-step)')
plt.xlabel('iteration'); plt.ylabel('weighted chi2'); plt.legend(); plt.tight_layout(); plt.show()

## Check 5 — Bridge-Theorem gauge: Φ W Φ† = I

In [ ]:
Q = w_orthonormalize(eig_full, w)
gram = (Q * w[None, :]) @ Q.conj().T
dev = np.max(np.abs(gram - np.eye(Q.shape[0])))
ok = dev < 1e-8
results['5 w-gauge P\u2020\u03a3\u207b\u00b9P = I'] = (ok, f'max |gram - I| = {dev:.3e}')
print(results['5 w-gauge P\u2020\u03a3\u207b\u00b9P = I'])

## Check 6 — PSD normalization round-trip (Fig 7 Brownian 184× bug hunt)

First, the notebook's own generator vs `PSDCalculator` (sanity). Then the
QP-simulator `noise_module.NoiseGenerator` — with a units finding from code review.

In [ ]:
noise_only = colored_noise(rng, J, N, FS, 2000)
f_est, J_est = calculate_psd(noise_only, sampling_frequency=FS)
ratio = np.median(J_est[1:-1] / J[1:-1])
ok = abs(ratio - 1.0) < 0.05
results['6a PSD round-trip (notebook generator)'] = (ok, f'median(J_est / J_true) = {ratio:.4f}')
print(results['6a PSD round-trip (notebook generator)'])

plt.figure(figsize=(5, 3))
plt.loglog(f[1:], J[1:], label='injected')
plt.loglog(f_est[1:], J_est[1:], '--', label='recovered (PSDCalculator)')
plt.xlabel('f (Hz)'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Round-trip through the QP-simulator noise module (Brownian = steepest PSD).
#
# IMPORTANT UNITS FINDING (code review 2026-06-12):
# noise_module.NoiseGenerator.build_psd() returns DFT-POWER |X_k|^2, NOT a
# physical one-sided PSD in A^2/Hz, despite its docstring. The physical
# one-sided PSD of the noise it generates is (QETpy convention):
#     J_phys[k]        = 2 * build_psd[k] / (fs * N)   (interior bins)
#     J_phys[0], [Nyq] =     build_psd[k] / (fs * N)
# If build_psd output (or 'noise_power' directly) is ever fed to
# OptimumFilter / the CRB formula as if it were A^2/Hz, the OF amplitude is
# unaffected (overall scale cancels) but predicted sigma_E and chi2 are off
# by a color-DEPENDENT factor: the analytic colors carry per-color band
# normalizations (_normalize_brownian ~ f_min, _normalize_white ~ 1/bandwidth)
# that differ by orders of magnitude. This is a prime suspect for the
# Fig 7 Brownian 184x discrepancy in the paper draft.
try:
    sys.path.insert(0, str(REPO_ROOT / 'QP_simulator'))
    from noise_module.NoiseGenerator import NoiseGenerator
    cfg = {'noise_type': 'brownian', 'noise_power': 1e-6, 'sampling_frequency': FS}
    gen = NoiseGenerator(cfg, seed=42)
    nz = np.stack([gen.generate_noise(N) for _ in range(1000)])
    f_q, J_est_q = calculate_psd(nz, sampling_frequency=FS)

    _, psd_dft = gen.build_psd(N)
    J_target = np.empty_like(psd_dft)
    J_target[1:-1] = 2.0 * psd_dft[1:-1] / (FS * N)
    J_target[0] = psd_dft[0] / (FS * N)
    J_target[-1] = psd_dft[-1] / (FS * N)

    ratio_q = np.median(J_est_q[2:-1] / J_target[2:-1])
    ok = abs(ratio_q - 1.0) < 0.1
    results['6b QP noise_module round-trip'] = (
        ok, f'median(recovered / converted target) = {ratio_q:.4f} '
            f'(raw build_psd is {FS * N / 2.0:.3g}x larger than physical PSD)')
    # variance check: total power should equal noise_power
    var_ratio = np.var(nz) / cfg['noise_power']
    print(f"variance / noise_power = {var_ratio:.4f} (expect ~1)")

    plt.figure(figsize=(5, 3))
    plt.loglog(f_q[1:], J_est_q[1:], label='recovered (PSDCalculator)')
    plt.loglog(f_q[1:], J_target[1:], '--', label='build_psd converted to A$^2$/Hz')
    plt.loglog(f_q[1:], psd_dft[1:], ':', label='raw build_psd output (wrong units)')
    plt.xlabel('f (Hz)'); plt.legend(fontsize=7); plt.tight_layout(); plt.show()
except Exception as e:
    results['6b QP noise_module round-trip'] = (False, f'FAILED: {e!r}')
print(results['6b QP noise_module round-trip'])

## Summary

In [ ]:
width = max(len(k) for k in results)
all_ok = True
for k, (ok, detail) in results.items():
    print(f"{'PASS' if ok else 'FAIL'}  {k.ljust(width)}  {detail}")
    all_ok &= ok
print('\n' + ('ALL CHECKS PASSED - the core code holds water.' if all_ok else 'SOME CHECKS FAILED - investigate above.'))